# B05 · S4 — Híbridos y guardarraíles (Pagarium)

**Objetivo (RA5-b/d/e):** extraer reglas de los datos (**FIGS**) y proteger un agente LLM con **guardarraíles** de reglas. Cierre: neuro-simbólico, AI Act y proyecto Pagarium.

> Práctica guiada de la S4 de los [apuntes](../apuntes.md).

## 1. FIGS redescubre la política latente de Pagarium

Generamos 4.000 pagos con una regla oculta y un 3 % de ruido; FIGS la recupera de forma legible.

In [ ]:
%pip install imodels scikit-learn
import numpy as np
from imodels import FIGSClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 4000
importe = rng.uniform(0, 2000, n)
antiguedad = rng.uniform(0, 24, n)
n_intentos = rng.integers(0, 6, n)

y = ((importe > 1000) & (antiguedad < 6)) | (n_intentos >= 4)
flip = rng.random(n) < 0.03
y = np.where(flip, ~y, y).astype(int)

X = np.column_stack([importe, antiguedad, n_intentos])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

clf = FIGSClassifier(max_rules=6)
clf.fit(X_tr, y_tr)
print("Accuracy:", round(clf.score(X_te, y_te), 3))
print(clf)

## 2. Guardarraíl de un agente LLM

Las reglas validan y acotan la propuesta del agente antes de ejecutarla.

In [ ]:
import re

def validar_accion_llm(texto):
    importes = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*EUR", texto)]
    if any(i > 1000 for i in importes):
        return False, "importe fuera de rango"
    if "cuenta no verificada" in texto.lower():
        return False, "destino no permitido"
    if re.search(r"transferir.*sin confirmar", texto, re.IGNORECASE):
        return False, "requiere confirmacion humana"
    return True, "ok"

for t in ["Transferir 500 EUR a la cuenta A",
          "Transferir 5000 EUR",
          "Pagar a cuenta no verificada"]:
    print(t, "->", validar_accion_llm(t))

## 3. Actividad — las dos capas del proyecto Pagarium

1. **Capa de negocio:** implementa la tabla DMN (S3) con `rule-engine` o `MicroMotor2`.
2. **Capa guardarraíl:** añade una regla nueva a `validar_accion_llm` (p. ej. bloquear divisas no permitidas).
3. Escribe un informe de 3 líneas: ¿usarías un motor de reglas aquí? Justifícalo con el benchmark.

In [ ]:
# TODO: capa de negocio + guardarrail + conclusion
...

## 4. Cierre — neuro-simbólico y AI Act

- **Neuro-simbólico:** red (aprende) + reglas (acotan y explican) → menos alucinaciones y más trazabilidad.
- **AI Act:** alto riesgo del Anexo III (scoring) aplazado a dic-2027 por el Ómnibus Digital; transparencia y alfabetización vigentes desde ago-2026. **Verifica el calendario antes de evaluar.**

**Para casa:** ¿por qué un guardarraíl de reglas mejora la confianza en un agente LLM?